# Fast-SCNN - Cityscapes Dataset

Treinamento do Fast-SCNN usando o dataset Cityscapes completo.

In [ ]:
import numpy
import torch
import os
import sys
!{sys.executable} -m pip uninstall -y numpy
!{sys.executable} -m pip install numpy==1.26.4
!{sys.executable} -m pip install matplotlib

print("Python executable:", sys.executable)
print("Numpy version:", numpy.__version__)
print("Numpy location:", numpy.__file__)
print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

# Teste de conversão numpy <-> torch
a = numpy.ones((2,2), dtype=numpy.float32)
print("Numpy array:", a)
try:
    t = torch.from_numpy(a)
    print("Torch tensor:", t)
except Exception as e:
    print("Erro ao converter numpy para torch:", e)

In [ ]:
# Dataset ULTRA-OTIMIZADO (sem conversão RGB->trainID, usa CrossEntropyLoss direto)
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CityscapesDatasetFast(Dataset):
    def __init__(self, cityscapes_root, split='train', img_height=128, img_width=256, max_samples=None):
        self.cityscapes_root = cityscapes_root
        self.split = split
        self.img_height = img_height
        self.img_width = img_width
        
        # Caminhos
        self.img_dir = os.path.join(cityscapes_root, 'leftImg8bit', split)
        self.gtfine_dir = os.path.join(cityscapes_root, 'gtFine', split)
        
        # Coleta imagens
        self.images = []
        
        self.annotations = []
        
        for city in sorted(os.listdir(self.img_dir)):
            city_img_dir = os.path.join(self.img_dir, city)
            city_gtfine_dir = os.path.join(self.gtfine_dir, city)
            
            if not os.path.isdir(city_img_dir):
                continue
            
            for img_file in sorted(os.listdir(city_img_dir)):
                if img_file.endswith('_leftImg8bit.png'):
                    img_path = os.path.join(city_img_dir, img_file)
                    base_name = img_file.replace('_leftImg8bit.png', '')
                    gtfine_file = f'{base_name}_gtFine_color.png'
                    gtfine_path = os.path.join(city_gtfine_dir, gtfine_file)
                    
                    if os.path.exists(gtfine_path):
                        self.images.append(img_path)
                        self.annotations.append(gtfine_path)
                        
                        if max_samples and len(self.images) >= max_samples:
                            break
            
            if max_samples and len(self.images) >= max_samples:
                break
        
        print(f'Dataset {split}: {len(self.images)} imagens')
        
        # LUT pré-calculada para conversão RÁPIDA
        self.lut = self._create_lut()
    
    def _create_lut(self):
        """Cria Look-Up Table para conversão RGB->trainID ULTRA-RÁPIDA"""
        lut = np.full((256, 256, 256), 255, dtype=np.uint8)  # 255 = ignore
        
        colors = {
            (128, 64, 128): 0, (244, 35, 232): 1, (70, 70, 70): 2, (102, 102, 156): 3,
            (190, 153, 153): 4, (153, 153, 153): 5, (250, 170, 30): 6, (220, 220, 0): 7,
            (107, 142, 35): 8, (152, 251, 152): 9, (70, 130, 180): 10, (220, 20, 60): 11,
            (255, 0, 0): 12, (0, 0, 142): 13, (0, 0, 70): 14, (0, 60, 100): 15,
            (0, 80, 100): 16, (0, 0, 230): 17, (119, 11, 32): 18
        }
        
        for (r, g, b), train_id in colors.items():
            lut[r, g, b] = train_id
        
        return lut
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Carrega imagem
        img = Image.open(self.images[idx]).convert('RGB')
        img = img.resize((1024, 512), Image.BILINEAR)  # AUMENTADO
        img = np.array(img, dtype=np.float32) / 255.0
        
        # Carrega máscara
        mask = Image.open(self.annotations[idx]).convert('RGB')
        mask = mask.resize((1024, 512), Image.NEAREST)  # AUMENTADO
        mask_rgb = np.array(mask, dtype=np.uint8)
        
        # Conversão RÁPIDA usando LUT
        trainid_mask = self.lut[mask_rgb[:,:,0], mask_rgb[:,:,1], mask_rgb[:,:,2]]
        
        # Converte para tensores
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(trainid_mask).long()
        
        return img_tensor, mask_tensor

# Caminho para o Cityscapes
cityscapes_root = r'd:\Documentos\Git\edge-segmentation-lab\city'

# 500 imagens, resolução 1024x512 (maior qualidade)
train_dataset = CityscapesDatasetFast(cityscapes_root, split='train', img_height=512, img_width=1024, max_samples=2000)

# Dataloader
train_loader = DataLoader(
    train_dataset, 
    batch_size=2,  # Reduzido para evitar estouro de memória
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print(f'\n⚡ Configuração otimizada:')
print(f'Dataset: {len(train_dataset)} imagens, {len(train_loader)} batches')
print(f'Resolução: 1024x512')
print(f'Batch size: 2')
print(f'Tempo estimado: ~20 ta vendo minutos/época, ~100 min total (5 épocas)')

In [ ]:
# Loss simplificada (CrossEntropyLoss já funciona com trainIDs)
import torch.nn as nn

criterion = nn.CrossEntropyLoss(ignore_index=255)

## Treinamento

In [ ]:
# Configuração GPU (OBRIGATÓRIO)
if not torch.cuda.is_available():
    raise RuntimeError('❌ GPU NÃO DISPONÍVEL! Este notebook requer GPU para funcionar.')

device = torch.device('cuda')
print(f'✓ Usando GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Memória disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'✓ Memória alocada: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')

In [ ]:
# Importar modelo Fast-SCNN
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src', 'models')))
from fast_scnn import FastSCNN

# Criar modelo
model = FastSCNN(num_classes=19, aux=False).to(device)
print(f'Modelo FastSCNN criado com {sum(p.numel() for p in model.parameters())/1e6:.2f}M parâmetros')

# Optimizer e Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
# Treinamento Fast-SCNN simplificado
import time
num_epochs = 15
best_loss = float('inf')
start_time = time.time()
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)[0]
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)
    scheduler.step(avg_train_loss)
    if avg_train_loss < best_loss:
        best_loss = avg_train_loss
        torch.save(model.state_dict(), 'fastscnn_best.pth')
        print(f'Epoca {epoch+1}/{num_epochs} | Loss: {avg_train_loss:.4f} ⭐')
    else:
        print(f'Epoca {epoch+1}/{num_epochs} | Loss: {avg_train_loss:.4f}')
print(f'Treinamento concluido em {((time.time()-start_time)/60):.1f} min | Melhor Loss: {best_loss:.4f}')

## Teste com Imagem Específica (Bielefeld)

In [ ]:
import matplotlib.pyplot as plt
import torch
from PIL import Image
import numpy as np

# Carregar imagem de teste específica de Bielefeld
test_img_path = r'D:\Documentos\Git\edge-segmentation-lab\city\leftImg8bit\test\bielefeld\bielefeld_000000_005068_leftImg8bit.png'
test_mask_path = r'D:\Documentos\Git\edge-segmentation-lab\city\gtFine\test\bielefeld\bielefeld_000000_005068_gtFine_color.png'

# Processar imagem (mesmo processamento do dataset)
img = Image.open(test_img_path).convert('RGB')
img_resized = img.resize((1024, 512))  # Mesma resolução do treinamento
img_np = np.array(img_resized, dtype=np.float32) / 255.0

# Processar máscara ground truth
mask = Image.open(test_mask_path).convert('RGB')  # FIX: garantir RGB
mask_resized = mask.resize((1024, 512), Image.NEAREST)
mask_rgb = np.array(mask_resized, dtype=np.uint8)

# Verificar shape
print(f'Shape da máscara: {mask_rgb.shape}')

# Converter RGB para trainId usando a paleta do Cityscapes
cityscapes_color_to_train = {
    (128, 64, 128): 0, (244, 35, 232): 1, (70, 70, 70): 2, (102, 102, 156): 3,
    (190, 153, 153): 4, (153, 153, 153): 5, (250, 170, 30): 6, (220, 220, 0): 7,
    (107, 142, 35): 8, (152, 251, 152): 9, (70, 130, 180): 10, (220, 20, 60): 11,
    (255, 0, 0): 12, (0, 0, 142): 13, (0, 0, 70): 14, (0, 60, 100): 15,
    (0, 80, 100): 16, (0, 0, 230): 17, (119, 11, 32): 18
}

h, w = mask_rgb.shape[:2]
trainid_mask = np.full((h, w), 255, dtype=np.uint8)
mask_flat = mask_rgb.reshape(-1, 3)

for color, train_id in cityscapes_color_to_train.items():
    matches = np.all(mask_flat == color, axis=1)
    trainid_mask.flat[matches] = train_id

# Paleta de cores para visualização
cityscapes_palette = [
    (128, 64, 128), (244, 35, 232), (70, 70, 70), (102, 102, 156),
    (190, 153, 153), (153, 153, 153), (250, 170, 30), (220, 220, 0),
    (107, 142, 35), (152, 251, 152), (70, 130, 180), (220, 20, 60),
    (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
    (0, 80, 100), (0, 0, 230), (119, 11, 32)
]

# Predição do modelo (se treinado)
try:
    model.eval()
    with torch.no_grad():
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
        pred = model(img_tensor)[0]
        pred_mask = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
    
    # Visualizar com predição
    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    
    axes[0, 0].imshow(img_resized)
    axes[0, 0].set_title('Imagem Original\nbielefeld_000000_005068', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(mask_rgb)
    axes[0, 1].set_title('Ground Truth (RGB)', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[1, 0].imshow(trainid_mask, cmap='tab20', vmin=0, vmax=18)
    axes[1, 0].set_title('Ground Truth (TrainIDs)', fontsize=12)
    axes[1, 0].axis('off')
    
    # Colore
    pred_colored = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in enumerate(cityscapes_palette):
        pred_colored[pred_mask == class_id] = color
    
    axes[1, 1].imshow(pred_colored)
    axes[1, 1].set_title('Predição do Modelo', fontsize=12)
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
except:
    # Se modelo não treinado, só mostrar ground truth
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img_resized)
    axes[0].set_title('Imagem Original\nbielefeld_000000_005068', fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(mask_rgb)
    axes[1].set_title('Ground Truth (RGB)', fontsize=12)
    axes[1].axis('off')
    
    axes[2].imshow(trainid_mask, cmap='tab20', vmin=0, vmax=18)
    axes[2].set_title('Ground Truth (TrainIDs 0-18)', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Mostrar estatísticas
unique_classes = np.unique(trainid_mask[trainid_mask != 255])
print(f'\nEstatísticas da imagem de teste:')
print(f'Tamanho: {img_resized.size}')
print(f'Classes presentes: {len(unique_classes)}')
print(f'IDs das classes: {unique_classes}')

# Nome das classes Cityscapes
class_names = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 
               'traffic light', 'traffic sign', 'vegetation', 'terrain', 
               'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train', 
               'motorcycle', 'bicycle']

print(f'\nClasses detectadas:')
for class_id in unique_classes:
    pixels = np.sum(trainid_mask == class_id)
    percentage = 100 * pixels / (h * w)
    print(f'  {class_id:2d} - {class_names[class_id]:15s}: {pixels:6d} pixels ({percentage:5.2f}%)')